In [0]:
%run "../01-setup/1.configure_access_to_cloud_storage"

In [0]:
%run "../01-setup/2.common_functions"

In [0]:
dbutils.widgets.text("p_data_source", "")
v_data_source = dbutils.widgets.get("p_data_source")

In [0]:
dbutils.widgets.text("p_file_date", "2025-01")
v_file_date = dbutils.widgets.get("p_file_date")
raw_race_path = f"{raw_folder_path}/{v_file_date}"

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType

In [0]:
races_schema_incremental = StructType([
    StructField("season", IntegerType(), False),
    StructField("round", IntegerType(), True),
    StructField("url", StringType(), True),
    StructField("raceName", StringType(), False),
    StructField("date", DateType(), True),
    StructField("circuitId", StringType(), True),
])


races_schema_static = StructType([
    StructField("raceId", IntegerType(), False),
    StructField("year", IntegerType(), True),
    StructField("round", IntegerType(), True),
    StructField("circuitId", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("date", DateType(), True),
    StructField("time", StringType(), True),
    StructField("url", StringType(), True)
])

if USE_INCREMENTAL:
    races_schema = races_schema_incremental
else:
    races_schema = races_schema_static


races_df = spark.read.csv(
    f"{raw_race_path}/races.csv",
    header=True,
    schema=races_schema
)
display(races_df)
races_df.printSchema()
races_df.describe().show()

In [0]:
from pyspark.sql.functions import current_timestamp, to_timestamp, concat, col, lit, try_to_timestamp

In [0]:
if USE_INCREMENTAL:
    races_final = (
        races_df
        .withColumn("race_timestamp", to_timestamp(col("date")))
        .withColumn("ingestion_date", current_timestamp())
        .withColumn("data_source", lit(v_data_source))
        .withColumn("file_date", lit(v_file_date))
        .withColumnRenamed("season", "race_year")
        .withColumnRenamed("raceName", "name")
        .withColumnRenamed("circuitId", "circuit_id")
        .select(
            "race_year",
            "round",
            "circuit_id",
            "name",
            "date",
            "race_timestamp",
            "url",
            "ingestion_date",
            "data_source",
            "file_date",
        )
    )
else:
    races_added_df = (
        races_df
        .withColumn(
            "race_timestamp",
            try_to_timestamp(
                concat(col("date"), lit(" "), col("time")),
                lit("yyyy-MM-dd HH:mm:ss"),
            ),
        )
        .withColumn("ingestion_date", current_timestamp())
    )

    races_final = (
        races_added_df
        .select("raceId", "year", "round", "circuitId", "name", "race_timestamp", "ingestion_date")
        .withColumnRenamed("raceId", "race_id")
        .withColumnRenamed("year", "race_year")
        .withColumnRenamed("circuitId", "circuit_id")
    )

display(races_final)


In [0]:
merge_delta_data(
    races_final.dropDuplicates(["race_year", "round"]),
    f"{processed_folder_path}/races",
    "tgt.race_year = src.race_year AND tgt.round = src.round",
    partition_columns=["race_year"],
    file_date_value=v_file_date,
)

In [0]:
df = spark.read.format("delta").load(f"{processed_folder_path}/races")
display(df.groupBy("file_date").count().orderBy("file_date"))

In [0]:
dbutils.notebook.exit("Success")